<h1 style="color:#D30982;text-align:center;">Chapter 5: State Preparation</h1>

<h2 style="color:#D30982;">What you'll learn</h2>

- The two isometry-based state preparers in the QDK and their settings
- Why a single Slater determinant (HF reference) requires only X gates — depth 1
- How circuit depth scales with the number of determinants in a multi-configurational state
- Why sparse isometry outperforms the regular (dense) isometry for CI/SCI wavefunctions
- How to export a prepared state circuit to QASM, QIR, Qiskit, and Q# formats

<h2 style="color:#D30982;">From wavefunction to quantum register</h2>

The previous chapters produced a classical wavefunction description: either a single Slater determinant (HF) or a superposition of many determinants (SCI). Before quantum phase estimation can run, this classical description must be encoded into an actual quantum circuit that maps |00...0⟩ to the target state.

This is **state preparation** (also called isometry synthesis). Both methods in the QDK are designed specifically for **CI/SCI wavefunctions** — states expressed as linear combinations of Slater determinants, exactly the representation produced by HF and all multi-configuration methods in Chapters 1–3. Other trial state representations (variational ansätze, tensor network states) are not supported by these preparers.

| Preparer | Strategy | When to use |
|---|---|---|
| `sparse_isometry_gf2x` | GF2X elimination on the sparse binary amplitude matrix | Any CI/SCI wavefunction; depth scales with number of determinants |
| `qiskit_regular_isometry` | Full dense unitary decomposition | Not practical for real CI/SCI states — depth scales with the full 2ⁿ Hilbert space regardless of sparsity |

The key distinction is **configuration count, not qubit count**: `sparse_isometry_gf2x` only touches the non-zero amplitudes in the wavefunction; `qiskit_regular_isometry` decomposes the full 2ⁿ-dimensional unitary even if the wavefunction has just a handful of determinants. For any realistic chemistry calculation, `sparse_isometry_gf2x` is the only practical choice.

In [ ]:
# Setup: SCF → valence → MP2 → autoCAS-EOS pipeline
from pathlib import Path
import numpy as np

import qdk_chemistry.plugins.pyscf
import qdk_chemistry.plugins.qiskit

from qdk_chemistry.data import Structure, Wavefunction
from qdk_chemistry._core.data import SciWavefunctionContainer
from qdk_chemistry.algorithms import available, create, print_settings
from qdk_chemistry.utils import Logger, compute_valence_space_parameters

Logger.set_global_level(Logger.LogLevel.off)

N2_XYZ = Path("../examples/data/stretched_n2.structure.xyz")
structure = Structure.from_xyz_file(N2_XYZ)

E_hf, wfn_hf = create("scf_solver").run(structure, charge=0, spin_multiplicity=1, basis_or_guess="cc-pvdz")
num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
wfn_val = create("active_space_selector", "qdk_valence",
                 num_active_electrons=num_val_e, num_active_orbitals=num_val_o).run(wfn_hf)
val_idx = wfn_val.get_orbitals().get_active_space_indices()
wfn_mp2 = create("orbital_localizer", "qdk_mp2_natural_orbitals").run(wfn_val, *val_idx)

loc_ham = create("hamiltonian_constructor").run(wfn_mp2.get_orbitals())
macis = create("multi_configuration_calculator", "macis_asci",
               calculate_one_rdm=True, calculate_two_rdm=True)
macis.settings().set("core_selection_strategy", "fixed")
n_a, n_b = wfn_mp2.get_active_num_electrons()
_, wfn_sci = macis.run(loc_ham, n_a, n_b)
wfn_eos = create("active_space_selector", "qdk_autocas_eos").run(wfn_sci)
# wfn_eos is not used in this chapter — it is carried forward to Chapter 6 (QPE setup)

print(f"HF energy: {E_hf:.6f} Hartree")
print(f"SCI wavefunction: {len(wfn_sci.get_active_determinants())} determinants, "
      f"{len(wfn_sci.get_orbitals().get_active_space_indices()[0])} orbitals")
print(f"autoCAS-EOS: {len(wfn_eos.get_active_determinants())} determinant, "
      f"{len(wfn_eos.get_orbitals().get_active_space_indices()[0])} orbitals")

<h2 style="color:#D30982;">Part 1: What state preparers are available?</h2>

`available("state_prep")` lists both preparers. `print_settings()` exposes three configurable parameters:

- `transpile` — if `True`, the synthesized circuit is recompiled into a specific native gate set after isometry synthesis. Leave `False` for simulation; set `True` when targeting hardware with a restricted gate vocabulary.
- `basis_gates` — the list of native gate names to target during transpilation (e.g., `["cx", "u3"]`). Only active when `transpile=True`; ignored otherwise.
- `dense_preparation_method` *(sparse only)* — controls how GF2X handles the dense sub-block of the amplitude matrix. The default works well for all wavefunctions seen in this course.

In [ ]:
# List available state preparers and their settings
print("Available state preparers:", available("state_prep"))
print()
for name in available("state_prep"):
    print(f"--- {name} ---")
    print_settings("state_prep", name)
    print()

# Two preparers:
#   sparse_isometry_gf2x     — GF2X-based sparse method; exploits few-determinant structure;
#                              depth scales with the number of non-zero amplitudes.
#   qiskit_regular_isometry  — Dense unitary decomposition; general-purpose but exponentially
#                              expensive; only practical for very small qubit counts.


<h2 style="color:#D30982;">Part 2: Single Slater determinant — HF reference</h2>

The simplest state to prepare is the Hartree-Fock reference: a single Slater determinant. Under Jordan-Wigner encoding, this is just a computational basis state with |1⟩ for each occupied spin-orbital and |0⟩ for each empty one. Preparing it from |00...0⟩ requires only **X gates** — no entanglement. Circuit depth is 1, regardless of system size.

Fill in the preparer name in the cell below, and then run it. 

In [ ]:
# Single Slater determinant: HF reference state preparation
sp_sparse = create("state_prep", "???")  # fill in: the sparse isometry preparer name
circ_hf   = sp_sparse.run(wfn_hf)

qc_hf = circ_hf.get_qiskit_circuit()
ops   = qc_hf.count_ops()
print(f"HF reference ({qc_hf.num_qubits} qubits):")
print(f"  Circuit depth : {qc_hf.depth()}")
print(f"  Total gates   : {sum(ops.values())}")
print(f"  CX gates      : {ops.get('cx', 0)}")
print()
print("QASM (first 8 lines):")
for line in circ_hf.get_qasm().split("\n")[:8]:
    print(f"  {line}")

# A single Slater determinant needs only X gates — one per occupied spin-orbital.
# Each X flips that spin-orbital from |0⟩ to |1⟩ to match the HF occupation pattern.
# No entangling gates: HF is a product state in the spin-orbital (Jordan-Wigner) basis.


<h2 style="color:#D30982;">Part 3: Scaling with number of determinants</h2>

Real chemistry wavefunctions have many determinants. The SCI result for N₂ has 3136 — requiring a depth-194,000 circuit if prepared exactly. In practice, the dominant determinants carry most of the wavefunction weight: keeping only the top-10 reduces depth to ~314 with minimal state error.

Depth grows roughly linearly with determinant count for `sparse_isometry_gf2x`: each additional determinant adds one binary column to the amplitude matrix that GF2X must synthesize into CNOT+X gates. Truncation is the primary lever for controlling hardware cost — at the expense of a small approximation in the trial state.

The `truncate_wavefunction` utility sorts determinants by |coefficient| and keeps the top-N, renormalized. Run the cell to see how depth scales.

In [ ]:
# Scaling of state preparation depth with number of determinants
def truncate_wavefunction(wfn, n_dets):
    """Keep the top-n_dets determinants by |coefficient|, renormalized."""
    dets   = wfn.get_active_determinants()
    coeffs = np.array([wfn.get_coefficient(d) for d in dets], dtype=float)
    order  = np.argsort(-np.abs(coeffs))
    top_dets   = [dets[i] for i in order[:n_dets]]
    top_coeffs = coeffs[order[:n_dets]]
    top_coeffs /= np.linalg.norm(top_coeffs)
    return Wavefunction(SciWavefunctionContainer(top_coeffs, top_dets, wfn.get_orbitals()))

sp = create("state_prep", "sparse_isometry_gf2x")

print(f"{'State':<32} {'Dets':>6} {'Qubits':>7} {'Depth':>8} {'CX gates':>9}")
print("-" * 66)
for n in [1, 2, 5, 10]:
    wfn_t = truncate_wavefunction(wfn_sci, n)
    circ  = sp.run(wfn_t)
    qc    = circ.get_qiskit_circuit()
    ops   = qc.count_ops()
    print(f"{'SCI top-'+str(n)+' dets':<32} {n:>6} {qc.num_qubits:>7} {qc.depth():>8} {ops.get('cx',0):>9}")

# Pre-computed for larger determinant counts (expensive to synthesize live)
print(f"{'SCI top-20 dets':<32} {'20':>6} {'16':>7} {'~640':>8} {'~490':>9}  ← pre-computed")
print()
print("Full SCI (3136 dets): depth ≈ 194,000, CX ≈ 131,000 — impractical for hardware.")
print("Circuit depth grows roughly linearly with determinant count for sparse_isometry_gf2x.")
print("Truncating to dominant determinants reduces depth at the cost of a small")
print("state approximation error. Top-10 dets capture the main correlation structure.")
wfn_10 = truncate_wavefunction(wfn_sci, 10)


<h2 style="color:#D30982;">Part 4: Sparse vs. regular isometry</h2>

`sparse_isometry_gf2x` uses <a href="https://journals.aps.org/pra/abstract/10.1103/PhysRevA.93.032318" target="_blank">GF2X</a> (Gaussian elimination over GF(2) augmented with X operations) to find a compact CNOT+X circuit. It exploits the sparsity of the CI/SCI wavefunction — only the non-zero amplitudes (the occupied determinants) contribute to the circuit. Depth grows linearly with the number of determinants, not with 2ⁿ.

`qiskit_regular_isometry` performs a full dense unitary decomposition. It does not exploit sparsity: its cost scales with the full 2ⁿ-dimensional Hilbert space regardless of how many determinants the wavefunction has. A wavefunction with 10 determinants on 16 qubits costs the same as a wavefunction with 65,536 determinants. For the same 10-determinant state: depth 314 (sparse) vs. 337,756 (regular).

The practical takeaway: the distinction between these two methods is **not about qubit count**, but it is about whether the method exploits the sparse structure of CI/SCI wavefunctions. For any chemistry calculation in this course, `sparse_isometry_gf2x` is the correct choice.

**Runtime note:** `qiskit_regular_isometry` decomposes a full 2¹⁶-dimensional unitary — expect ~5 minutes on most laptops.

Fill in the preparer name to complete the comparison, in the cell below. 

In [ ]:
# Sparse vs. regular isometry on the same 10-determinant state
sp_sparse  = create("state_prep", "sparse_isometry_gf2x")
sp_regular = create("state_prep", "???")  # fill in: the dense isometry preparer name
# Note: sp_regular.run() decomposes a full 2^16-dimensional unitary (~5 minutes).
# Its result is pre-computed and shown below; only the sparse preparer runs here.

print(f"{'Preparer':<30} {'Depth':>8} {'CX gates':>10} {'Total gates':>12}")
print("-" * 64)

# Run sparse isometry (fast — depth scales with determinant count, not Hilbert space)
circ = sp_sparse.run(wfn_10)
qc   = circ.get_qiskit_circuit()
ops  = qc.count_ops()
print(f"{'sparse_isometry_gf2x':<30} {qc.depth():>8} {ops.get('cx',0):>10} {sum(ops.values()):>12}")

# Regular isometry result (pre-computed — run sp_regular.run(wfn_10) to verify, takes ~5 minutes)
print(f"{'qiskit_regular_isometry':<30} {'337756':>8} {'262120':>10} {'337801':>12}  ← pre-computed")

print()
print("sparse_isometry_gf2x exploits the few-determinant structure: GF2X elimination")
print("finds a short CNOT+X sequence that encodes the superposition. For this 10-det")
print("state: depth ~314 vs 337,756 — a ~1000x reduction in circuit depth.")
print("For near-term hardware, always use the sparse preparer.")


# Question: chem-state-prep-cqapy8

> Question not found (HTTP 404)


<h2 style="color:#D30982;">Part 5: Refining the trial state</h2>

`truncate_wavefunction` (Part 3) simply renormalizes the top-N SCI coefficients — the coefficients are frozen at their SCI values, just rescaled. The **projected multi-configuration calculator** (PMC) does better: given the same N determinants, it re-solves the Hamiltonian in that subspace and re-optimizes the coefficients. The result is a lower variational energy and, critically, higher **fidelity** with the true ground state.

Fidelity = $|\langle \psi_{pmc} | \psi_{pmc} \rangle |^2$ is the probability that QPE (Chapter 6) collapses to the ground state on the first shot (<a href="https://arxiv.org/abs/quant-ph/0005055" target="_blank">Brassard et al., 2002</a>). A trial state with fidelity F succeeds with probability F — making fidelity the direct link between state preparation quality and QPE reliability.

In the cell below, fill in the "???" and then answer the subsequent question.

In [ ]:
# Projected multi-configuration refinement and trial state fidelity
# truncate_wavefunction (above) renormalizes the top-N SCI coefficients as-is.
# The projected multi-configuration calculator (PMC) re-solves the Hamiltonian
# in the subspace of those N determinants, optimizing the coefficients — giving
# a lower variational energy and better overlap with the full SCI state.
#
# Fidelity = |⟨ψ_pmc | ψ_sci⟩|² = probability that QPE (Chapter 6) collapses
# to the ground state on the first shot. Higher fidelity → more reliable QPE.

pmc = create("projected_multi_configuration_calculator", "???")  # fill in: PMC algorithm name

print(f"{'N dets':<8} {'Fidelity':>10} {'Circ depth':>12}")
print("-" * 34)
for n in [1, 2, 5, 10]:
    top_dets = wfn_sci.get_top_determinants(max_determinants=n)
    det_list = list(top_dets.keys())
    _, wfn_pmc = pmc.run(loc_ham, det_list)

    c_sci = np.array([wfn_sci.get_coefficient(d) for d in det_list])
    c_pmc = np.array([wfn_pmc.get_coefficient(d) for d in det_list])
    fidelity = float(np.abs(np.vdot(c_pmc, c_sci)) ** 2)
    depth = sp.run(wfn_pmc).get_qiskit_circuit().depth()

    print(f"{n:<8} {fidelity:>10.4f} {depth:>12}")

print()
print("PMC re-optimizes coefficients within the subspace — unlike simple truncation.")
print("Fidelity → 1.0 as N grows: more determinants, better QPE success probability.")
# Carry the 10-det PMC-refined state forward to Chapter 6
top_10 = list(wfn_sci.get_top_determinants(max_determinants=10).keys())
_, wfn_pmc_10 = pmc.run(loc_ham, top_10)

# Question: state-prep-frq-660que

> Question not found (HTTP 404)


<h2 style="color:#D30982;">Part 6: Circuit export formats</h2>

A prepared `Circuit` object can be exported in four formats:
- `get_qasm()` — OpenQASM 3.0 string (hardware-agnostic, human-readable)
- `get_qiskit_circuit()` — Qiskit `QuantumCircuit` (for noise simulation or transpilation)
- `get_qsharp_circuit()` — Q# circuit (for QDK resource estimation)
- `get_qir()` — <a href="https://github.com/qir-alliance" target="_blank">QIR</a> bitcode (compiled, hardware-ready for Azure Quantum)

QIR (Quantum Intermediate Representation) is the native compiled format for Azure Quantum hardware — the format you submit when running on real quantum hardware.

Typical workflow: inspect and validate via QASM → simulate noise via Qiskit → project hardware costs via Q# resource estimation → submit compiled QIR to Azure Quantum. The state preparation circuit produced here feeds directly into the IQPE iteration circuits in Chapter 6.

In [ ]:
# Circuit export: QASM, Qiskit, Q#, and QIR
circ_10 = create("state_prep", "sparse_isometry_gf2x").run(wfn_10)

print("Export methods on a Circuit object:")
print("  get_qasm()           → OpenQASM 3.0 string (hardware-agnostic)")
print("  get_qiskit_circuit() → Qiskit QuantumCircuit (simulation, noise modeling)")
print("  get_qsharp_circuit() → Q# circuit (QDK resource estimation)")
print("  get_qir()            → QIR bitcode (compiled, hardware-ready for Azure Quantum)")
print()
qasm_str = circ_10.get_qasm()
print("QASM excerpt (first 8 lines of 10-det state circuit):")
for line in qasm_str.split("\n")[:8]:
    print(f"  {line}")
print("  ...")
print()

# Inspect the Qiskit circuit directly
qc = circ_10.get_qiskit_circuit()
ops = qc.count_ops()
print(f"Qiskit QuantumCircuit: {qc.num_qubits} qubits, depth={qc.depth()}, "
      f"CX gates={ops.get('cx', 0)}, total gates={sum(ops.values())}")
print("→ Matches the sparse_isometry depth/CX count from Part 3.")

print()
print("→ Carry forward: The sparse state preparation approach (sparse_isometry_gf2x)")
print("  and the autoCAS-EOS active space (from Ch.4) feed Chapter 6 (QPE).")


# Multiple Choice Question: chem-state-prep-future-75in0b

The full SCI wavefunction (3136 determinants) produces a state preparation circuit with depth ~194,000. What is the most practical strategy for near-term hardware?

## Choices

A. Truncate to the dominant determinants (top-N by |coefficient|)
B. Skip state preparation and start from |000...0⟩
C. Increase the basis set to reduce the number of determinants
D. Use qiskit_regular_isometry instead — it handles large systems better


<h2 style="color:#D30982;">Summary</h2>

In this chapter you:
- Listed the two state preparers (`sparse_isometry_gf2x`, `qiskit_regular_isometry`) and their settings
- Prepared the HF reference state and confirmed it produces a depth-1 X-gate circuit
- Built a `truncate_wavefunction` utility and measured how circuit depth grows with determinant count
- Compared sparse vs. regular isometry on a 10-determinant state (depth 314 vs. 337,756)
- Refined the 10-determinant trial state with PMC and measured fidelity with the full SCI state
- Exported the prepared circuit to QASM, Qiskit, Q#, and QIR formats

**Key pattern (state prep + refinement):**
```python
from qdk_chemistry._core.data import SciWavefunctionContainer
from qdk_chemistry.data import Wavefunction

# Truncate to top-N determinants
wfn_10 = truncate_wavefunction(wfn_sci, 10)

# Prepare circuit
sp = create("state_prep", "sparse_isometry_gf2x")
circ = sp.run(wfn_10)

# Inspect and export
qc = circ.get_qiskit_circuit()
print(qc.count_ops())
qasm = circ.get_qasm()
```

The `wfn_10` and `circ_10` are carried forward into Chapter 6 (QPE).
